# KorQuAD QLoRA 파인튜닝 (Qwen2.5-1.5B-Instruct, Colab 무료 T4)

셀을 S0 → S4 순서로 실행한다. 단일 진실은 `src/` 스크립트이며, 이 노트북은
스크립트 호출 중심으로 구성된다. 각 셀 주석에 대응 ISC ID를 명기했다.


## S0 — 환경·설치·스모크

GPU 확인, 의존성 고정 설치, import·4bit 로딩·1-step 학습 스모크까지.
1-step 스모크는 fp16 강제와 SFTTrainer의 bf16 자동 캐스팅이 T4에서
충돌하는지(train.py 상단 주석의 미확정 리스크)를 실측하는 프로브다.


In [ ]:
# ISC-0.1: Tesla T4 확인
!nvidia-smi

In [ ]:
# 리포 클론 (이미 클론돼 있으면 이 셀은 건너뛴다)
!git clone https://github.com/calintzy/korquad-qlora-lab.git
%cd korquad-qlora-lab

In [ ]:
# 의존성 고정 버전 설치 (torch는 Colab 프리인스톨 사용)
!pip install -q -r requirements.txt

In [ ]:
# ISC-0.2: 핵심 패키지 import 검증
import transformers, peft, bitsandbytes, trl, datasets, accelerate
print("transformers", transformers.__version__)
print("peft", peft.__version__)
print("bitsandbytes", bitsandbytes.__version__)
print("trl", trl.__version__)
print("datasets", datasets.__version__)
print("accelerate", accelerate.__version__)

In [ ]:
# Drive 마운트 (어댑터를 Drive에 저장해 세션 종료 후에도 유지)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ISC-0.3: 4bit 로딩 스모크 + VRAM 확인
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
m = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct", quantization_config=bnb, device_map="auto"
)
print("loaded. VRAM(GB):", round(torch.cuda.memory_allocated() / 1e9, 2))
del m
torch.cuda.empty_cache()

In [ ]:
# 1-step 학습 스모크 (dtype 크래시·템플릿 자동 교체 실측)
!python src/train.py --output-dir /tmp/smoke --max-steps 1

## S1 — 채점기 자가검증 + 데이터 로드

공식 문자 단위 EM/F1 채점기가 올바른지(ISC-1.2/1.3) 확인하고,
KorQuAD split 크기를 출력한다(ISC-1.1).


In [ ]:
# ISC-1.2 / ISC-1.3: 채점기 자가검증 (ALL PASS 기대)
!python tests/test_scorer.py

In [ ]:
# ISC-1.1: 데이터 로드 및 split 크기 확인
import sys; sys.path.insert(0, "src")
import data
ds = data.load_korquad()
print("train:", len(ds["train"]))
print("validation:", len(ds["validation"]))

## S2 — 제로샷 평가 (before)

파인튜닝 전 베이스 모델 성능을 측정한다.

**서브셋 결정 가이드**: 첫 배치 몇 개의 속도를 보고 판단한다.
- 전체(약 5,700문항)가 무료 T4 세션 시간 내에 끝날 것 같으면 `--subset-n` 없이 전체.
- 느리면 `--subset-n 1000`으로 고정 시드 서브셋을 만든다(인덱스는
  `results/subset_indices.json`에 저장돼 after 평가와 동일 문항을 쓴다).
- **전체 평가로 되돌리려면 `results/subset_indices.json`을 삭제해야 한다**
  (파일 존재가 서브셋 모드보다 우선한다 — 파일이 있으면 `--subset-n`을 빼도
  저장된 인덱스가 그대로 적용된다).


In [ ]:
# ISC-2.1: 제로샷 평가 → results/results_before.json
# 전체로 돌리려면 --subset-n 을 제거한다.
!python src/eval_run.py \
  --output results/results_before.json \
  --subset-n 1000 \
  --predictions-out results/predictions_before.json

## S3 — QLoRA 본 학습

어댑터는 Drive 경로에 저장한다. 아래 `ADAPTER_DIR`를 필요 시 수정한다.
세션이 끊기면 재개 셀을 실행한다(같은 output-dir + `--resume`).


In [ ]:
ADAPTER_DIR = "/content/drive/MyDrive/korquad-qlora-lab/adapter"
print("adapter dir:", ADAPTER_DIR)

In [ ]:
# 본 학습 (1 에폭)
# ISC-3.2: 학습 로그의 loss가 실제로 하락하는지 확인 — 하락이 없으면 학습이
# 성립하지 않은 것이다(프롬프트 마스킹·dtype·데이터 포맷 문제를 의심).
!python src/train.py --output-dir "$ADAPTER_DIR" --epochs 1

In [ ]:
# 재개 셀 (세션이 끊겼을 때만 실행)
!python src/train.py --output-dir "$ADAPTER_DIR" --epochs 1 --resume

## S4 — 파인튜닝 후 평가 (after) + before/after 표

before와 동일한 서브셋(results/subset_indices.json)으로 평가한다.
유일한 차이는 `--adapter`뿐이다.


In [ ]:
# ISC-4.1: 파인튜닝 후 평가 → results/results_after.json
!python src/eval_run.py \
  --adapter "$ADAPTER_DIR" \
  --output results/results_after.json \
  --predictions-out results/predictions_after.json

In [ ]:
# ISC-4.2: before/after 표 출력
import json

with open("results/results_before.json") as f:
    before = json.load(f)
with open("results/results_after.json") as f:
    after = json.load(f)

print(f"{'구분':<10}{'EM':>10}{'F1':>10}{'n':>8}")
print(f"{'before':<10}{before['em']:>10.2f}{before['f1']:>10.2f}{before['n']:>8}")
print(f"{'after':<10}{after['em']:>10.2f}{after['f1']:>10.2f}{after['n']:>8}")
print(f"{'delta':<10}{after['em']-before['em']:>10.2f}{after['f1']-before['f1']:>10.2f}")